**Импорты**

In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=74ce28da4f1e71e2418810416e09c8ff84bb56605700fed4887066b28d79d9fa
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import os, glob, zipfile
import numpy as np
from collections import defaultdict

from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score

**Константы**

In [3]:
WORKING_DIR = "/kaggle/working/"
os.makedirs(WORKING_DIR, exist_ok=True)

DATA_DIR = os.path.join(WORKING_DIR, "data")
RESULTS_DIR = os.path.join(WORKING_DIR, "results")
HF_CACHE = os.path.join(WORKING_DIR, "hf_cache")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE, "transformers")

ROOT_DIR = "/kaggle/input/datasets/chepurchenko/ruacter/ru-ACTER/ru"

MODEL_NAME_RUBERT = "ai-forever/ruBert-base"
MODEL_NAME_RUROBERTA = "ai-forever/ruRoberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 10
SEED = 42
VAL_SIZE = 0.1
FP16 = True

DOMAIN_MAP = {
    "corp": "corruption",
    "equi": "equestrian",
    "htfl": "heart_failure",
    "wind": "wind_energy",
}

label2id = {"O": 0, "B": 1, "I": 2}
id2label = {v: k for k, v in label2id.items()}

**Загрузка всех BIO‑файлов**

In [4]:
def read_bio_file(filepath):
    sentences, labels = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        cur_tokens, cur_labels = [], []
        for line in f:
            line = line.strip()
            if not line:
                if cur_tokens:
                    sentences.append(cur_tokens)
                    labels.append(cur_labels)
                    cur_tokens, cur_labels = [], []
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            token, label = parts
            cur_tokens.append(token)
            cur_labels.append(label)
        if cur_tokens:
            sentences.append(cur_tokens)
            labels.append(cur_labels)
    return sentences, labels

bio_files = glob.glob(os.path.join(ROOT_DIR, "**/iob_annotations/with_named_entities/*.txt"), recursive=True)

domain_data = defaultdict(list)
for fpath in bio_files:
    domain = None
    for part in fpath.split(os.sep):
        if part in DOMAIN_MAP:
            domain = DOMAIN_MAP[part]
            break
    if domain is None:
        continue
    sents, labs = read_bio_file(fpath)
    for tokens, labels in zip(sents, labs):
        domain_data[domain].append({"tokens": tokens, "labels": labels})

datasets = {dom: Dataset.from_list(examples) for dom, examples in domain_data.items()}

# Пример
first_dom = list(datasets.keys())[0]
print(f"Домен: {first_dom}, предложений: {len(datasets[first_dom])}")
sample = datasets[first_dom][0]
print("Токены:", sample["tokens"][:10])
print("Метки :", sample["labels"][:10])

Домен: heart_failure, предложений: 2433
Токены: ['Лечение', 'сердечной', 'недостаточности', 'с', 'сохраненной', 'фракцией', 'выброса', '.']
Метки : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


# Эксперименты

**Токенизатор и выравнивание меток**

In [5]:
def tokenize_and_align(examples, tokenizer, label2id, max_length=MAX_LENGTH, stride=128):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        return_overflowing_tokens=True,
        stride=stride,
        return_tensors=None,
    )

    overflow_to_sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = []

    for batch_i, sample_i in enumerate(overflow_to_sample_mapping):
        label = examples["labels"][sample_i]
        word_ids = tokenized_inputs.word_ids(batch_index=batch_i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id.get(label[word_idx], -100))
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

**LODO-разбиения**

In [6]:
def create_lodo_splits(domain_data, test_domain, val_size=VAL_SIZE):
    test_ds = domain_data[test_domain]
    train_doms = [d for d in domain_data if d != test_domain]
    train_ds = concatenate_datasets([domain_data[d] for d in train_doms])
    train_val = train_ds.train_test_split(test_size=val_size, seed=SEED)
    return DatasetDict({
        "train": train_val["train"],
        "validation": train_val["test"],
        "test": test_ds,
    })

def prepare_lodo_experiment(domain_data, test_domain, tokenizer, label2id):
    splits = create_lodo_splits(domain_data, test_domain)
    tokenized = {}
    for split_name, ds in splits.items():
        ds = ds.map(
            lambda x: tokenize_and_align(x, tokenizer, label2id),
            batched=True,
            remove_columns=ds.column_names,
        )
        tokenized[split_name] = ds
    return DatasetDict(tokenized)

**Метрики**

In [7]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    y_true, y_pred = [], []
    for pseq, lseq in zip(preds, labels):
        t, p = [], []
        for pi, li in zip(pseq, lseq):
            if li == -100:
                continue
            t.append(id2label[li])
            p.append(id2label[pi])
        y_true.append(t)
        y_pred.append(p)

    return {
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
    }

**Обучение для одного LODO-домена**

In [8]:
def run_lodo_experiment(domain_data, test_domain, tokenizer, model_name, label2id, id2label):
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

    dataset = prepare_lodo_experiment(domain_data, test_domain, tokenizer, label2id)

    training_args = TrainingArguments(
        output_dir=os.path.join(RESULTS_DIR, f"lodo_{test_domain}_{model_name.replace('/', '_')}"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        num_train_epochs=10,
        weight_decay=0.01,
        logging_steps=20,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        fp16=FP16,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    test_results = trainer.evaluate(dataset["test"])
    return test_results

**Полный кросс‑доменный цикл**

RuBERT-base

In [18]:
tokenizer_rubert = AutoTokenizer.from_pretrained("ai-forever/ruBert-base", add_prefix_space=True)
results_rubert = {}
for test_domain in DOMAIN_MAP.values():
    print(f"\n===== [ruBert] Тестовый домен: {test_domain} =====")
    metrics = run_lodo_experiment(
        datasets, test_domain, tokenizer_rubert, "ai-forever/ruBert-base", label2id, id2label
    )
    results_rubert[test_domain] = metrics
    print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


===== [ruBert] Тестовый домен: corruption =====


pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

F1=0.3123, Precision=0.3377, Recall=0.2905


In [19]:
avg_f1 = np.mean([r["eval_f1"] for r in results_rubert.values()])
avg_prec = np.mean([r["eval_precision"] for r in results_rubert.values()])
avg_rec = np.mean([r["eval_recall"] for r in results_rubert.values()])
print(f"Средние метрики")
print(f"F1={avg_f1:.4f}, Precision={avg_prec:.4f}, Recall={avg_rec:.4f}")

Средние метрики
F1=0.2806, Precision=0.5185, Recall=0.2057


ruRoberta-large

In [9]:
tokenizer_ruroberta = AutoTokenizer.from_pretrained("ai-forever/ruRoberta-large", add_prefix_space=True)
results_ruroberta = {}
for test_domain in DOMAIN_MAP.values():
    print(f"\n===== [ruroberta] Тестовый домен: {test_domain} =====")
    metrics = run_lodo_experiment(
        datasets, test_domain, tokenizer_ruroberta, "ai-forever/ruRoberta-large", label2id, id2label
    )
    results_ruroberta[test_domain] = metrics
    print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]


===== [ruroberta] Тестовый домен: corruption =====


pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.438143,0.371549,0.702979,0.657993,0.754568
2,0.294531,0.364560,0.730684,0.668012,0.806334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 2479743488 vs 2479743376

In [13]:
tokenizer_ruroberta = AutoTokenizer.from_pretrained("ai-forever/ruRoberta-large", add_prefix_space=True)

print(f"\n===== [ruroberta] Тестовый домен: {"wind_energy"} =====")
metrics = run_lodo_experiment(
    datasets, "wind_energy", tokenizer_ruroberta, "ai-forever/ruRoberta-large", label2id, id2label
)
results_ruroberta["wind_energy"] = metrics
print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")



===== [ruroberta] Тестовый домен: wind_energy =====


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: ai-forever/ruRoberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/6774 [00:00<?, ? examples/s]

Map:   0%|          | 0/753 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

F1=0.2229, Precision=0.2939, Recall=0.1796


In [14]:
avg_f1 = np.mean([r["eval_f1"] for r in results_ruroberta.values()])
avg_prec = np.mean([r["eval_precision"] for r in results_ruroberta.values()])
avg_rec = np.mean([r["eval_recall"] for r in results_ruroberta.values()])
print(f"Средние метрики")
print(f"F1={avg_f1:.4f}, Precision={avg_prec:.4f}, Recall={avg_rec:.4f}")

Средние метрики
F1=0.2640, Precision=0.5032, Recall=0.1839


**In-domain эксперименты**

In [9]:
def run_in_domain_experiment(model_name, tokenizer, domain_data, domain, label2id, id2label):
    """
    Обучение и оценка внутри одного домена.
    Данные домена делятся на train (80%), validation (10%), test (10%).
    """
    ds = domain_data[domain]
    # Разбиение: 80% train, 10% val, 10% test
    train_val_test = ds.train_test_split(test_size=0.2, seed=SEED)
    test_ds = train_val_test['test']
    train_val = train_val_test['train'].train_test_split(test_size=0.125, seed=SEED)
    train_ds = train_val['train']
    val_ds = train_val['test']

    # Токенизация каждого сплита
    train_ds = train_ds.map(
        lambda x: tokenize_and_align(x, tokenizer, label2id, MAX_LENGTH),
        batched=True, remove_columns=train_ds.column_names
    )
    val_ds = val_ds.map(
        lambda x: tokenize_and_align(x, tokenizer, label2id, MAX_LENGTH),
        batched=True, remove_columns=val_ds.column_names
    )
    test_ds = test_ds.map(
        lambda x: tokenize_and_align(x, tokenizer, label2id, MAX_LENGTH),
        batched=True, remove_columns=test_ds.column_names
    )


    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(RESULTS_DIR, f"in_domain_{domain}_{model_name.replace('/', '_')}"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        num_train_epochs=10,
        weight_decay=0.01,
        logging_steps=20,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        fp16=FP16,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    test_results = trainer.evaluate(test_ds)
    return test_results

In [20]:
print("\n===== IN-DOMAIN: RuBERT-base =====")
tokenizer_rubert = AutoTokenizer.from_pretrained(MODEL_NAME_RUBERT, add_prefix_space=True)
results_in_rubert = {}
for domain in DOMAIN_MAP.values():
    print(f"\n--- Домен: {domain} ---")
    metrics = run_in_domain_experiment(
        MODEL_NAME_RUBERT, tokenizer_rubert, datasets, domain, label2id, id2label
    )
    results_in_rubert[domain] = metrics
    print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

avg_f1 = np.mean([r['eval_f1'] for r in results_in_rubert.values()])
print(f"\nСреднее по доменам: F1={avg_f1:.4f}")


===== IN-DOMAIN: RuBERT-base =====

--- Домен: corruption ---


Map:   0%|          | 0/1402 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

Map:   0%|          | 0/401 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

F1=0.6073, Precision=0.6082, Recall=0.6064

Среднее по доменам: F1=0.7070


In [14]:
print("\n===== IN-DOMAIN: ruRoberta-large =====")
tokenizer_ruroberta = AutoTokenizer.from_pretrained(MODEL_NAME_RUROBERTA, add_prefix_space=True)
results_in_ruroberta = {}
for domain in DOMAIN_MAP.values():
    print(f"\n--- Домен: {domain} ---")
    metrics = run_in_domain_experiment(
        MODEL_NAME_RUROBERTA, tokenizer_ruroberta, datasets, domain, label2id, id2label
    )
    results_in_ruroberta[domain] = metrics
    print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

avg_f1 = np.mean([r['eval_f1'] for r in results_in_ruroberta.values()])
print(f"\nСреднее по доменам: F1={avg_f1:.4f}")


===== IN-DOMAIN: ruRoberta-large =====


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]


--- Домен: corruption ---


...

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.270937,0.353205,0.492625,0.510703,0.475783
2,0.256240,0.330355,0.573888,0.578035,0.569801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 2479743488 vs 2479743376

In [11]:
print("\n===== IN-DOMAIN: ruRoberta-large =====")
tokenizer_ruroberta = AutoTokenizer.from_pretrained(MODEL_NAME_RUROBERTA, add_prefix_space=True)

domain = 'wind_energy'
print(f"\n--- Домен: {domain} ---")
metrics = run_in_domain_experiment(
    MODEL_NAME_RUROBERTA, tokenizer_ruroberta, datasets, domain, label2id, id2label
)
# results_in_ruroberta[domain] = metrics
print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

# avg_f1 = np.mean([r['eval_f1'] for r in results_in_ruroberta.values()])
# print(f"\nСреднее по доменам: F1={avg_f1:.4f}")


===== IN-DOMAIN: ruRoberta-large =====

--- Домен: wind_energy ---


Map:   0%|          | 0/4646 [00:00<?, ? examples/s]

Map:   0%|          | 0/664 [00:00<?, ? examples/s]

Map:   0%|          | 0/1328 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

F1=0.6530, Precision=0.5969, Recall=0.7208


# Анализ ошибок

In [40]:
MODELS_ROOT = "/kaggle/input/models/chepurchenko/ate-berts-/transformers/default/1/models"

In [53]:
def error_analysis(model_ckpt, tokenizer, test_dataset, domain_name, label2id, id2label, max_length, batch_size, num_examples=3):
    model = AutoModelForTokenClassification.from_pretrained(model_ckpt)
    
    tokenized_test = test_dataset.map(
        lambda x: tokenize_and_align(x, tokenizer, label2id, max_length),
        batched=True,
        remove_columns=test_dataset.column_names
    )
    
    data_collator = DataCollatorForTokenClassification(tokenizer)
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="./tmp_err", per_device_eval_batch_size=batch_size*2, report_to="none"),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    predictions, labels, _ = trainer.predict(tokenized_test)
    pred_labels = np.argmax(predictions, axis=2)
    
    true_sequences, pred_sequences = [], []
    for i in range(len(labels)):
        true_seq, pred_seq = [], []
        for j in range(len(labels[i])):
            if labels[i][j] != -100:
                true_seq.append(id2label[labels[i][j]])
                pred_seq.append(id2label[pred_labels[i][j]])
        true_sequences.append(true_seq)
        pred_sequences.append(pred_seq)
    
    def extract_terms(seq, tokens):
        terms = []
        start = None
        for i, label in enumerate(seq):
            if label == "B":
                if start is not None:
                    terms.append(" ".join(tokens[start:i]))
                start = i
            elif label == "I":
                if start is None:
                    start = i
            else:
                if start is not None:
                    terms.append(" ".join(tokens[start:i]))
                    start = None
        if start is not None:
            terms.append(" ".join(tokens[start:len(seq)]))
        return terms
    
    true_terms, pred_terms = [], []
    for i, sample in enumerate(test_dataset):
        tokens = sample["tokens"]
        true_terms.append(extract_terms(true_sequences[i], tokens))
        pred_terms.append(extract_terms(pred_sequences[i], tokens))
    
    all_true = [(i, t) for i, t_list in enumerate(true_terms) for t in t_list]
    all_pred = [(i, p) for i, p_list in enumerate(pred_terms) for p in p_list]
    
    matched_true, matched_pred = set(), set()
    for t_idx, (t_sent, t_term) in enumerate(all_true):
        for p_idx, (p_sent, p_term) in enumerate(all_pred):
            if t_sent == p_sent and t_term == p_term:
                matched_true.add(t_idx)
                matched_pred.add(p_idx)
                break
    
    fp_indices = [j for j in range(len(all_pred)) if j not in matched_pred]
    fn_indices = [i for i in range(len(all_true)) if i not in matched_true]
    
    tp_terms = [all_true[i][1] for i in matched_true]
    fp_terms = [all_pred[j][1] for j in fp_indices]
    fn_terms = [all_true[i][1] for i in fn_indices]
    
    print(f"\n--- Анализ ошибок: {domain_name} ---")
    print(f"TP: {len(tp_terms)}, FP: {len(fp_terms)}, FN: {len(fn_terms)}")
    
    from collections import Counter
    def len_dist(terms):
        return Counter(len(t.split()) for t in terms)
    
    print("\nРаспределение длин TP:")
    for l, c in sorted(len_dist(tp_terms).items()):
        print(f"  длина {l}: {c}")
    print("\nРаспределение длин FP:")
    for l, c in sorted(len_dist(fp_terms).items()):
        print(f"  длина {l}: {c}")
    print("\nРаспределение длин FN:")
    for l, c in sorted(len_dist(fn_terms).items()):
        print(f"  длина {l}: {c}")
    
    print("\nПримеры False Positives:")
    for t in fp_terms[:num_examples]:
        print(f"  '{t}'")
    print("\nПримеры False Negatives:")
    for t in fn_terms[:num_examples]:
        print(f"  '{t}'")

In [55]:
from sklearn.model_selection import train_test_split

TEST_DOMAIN = "heart_failure"
ds_heart = datasets[TEST_DOMAIN]
train_val_test = ds_heart.train_test_split(test_size=0.2, seed=SEED)
test_ds_heart = train_val_test['test']

# LODO ruBERT-base
MODEL_CKPT_RUBERT_LODO = os.path.join(MODELS_ROOT, "lodo", "rubert", TEST_DOMAIN)
error_analysis(
    model_ckpt=MODEL_CKPT_RUBERT_LODO,
    tokenizer=tokenizer_rubert,
    test_dataset=datasets[TEST_DOMAIN],
    domain_name="ruBERT-base LODO heart_failure",
    label2id=label2id,
    id2label=id2label,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)

# LODO ruRoBERTa-large
MODEL_CKPT_RUROBERTA_LODO = os.path.join(MODELS_ROOT, "lodo", "ruroberta", TEST_DOMAIN)
error_analysis(
    model_ckpt=MODEL_CKPT_RUROBERTA_LODO,
    tokenizer=tokenizer_ruroberta,
    test_dataset=datasets[TEST_DOMAIN],
    domain_name="ruRoBERTa-large LODO heart_failure",
    label2id=label2id,
    id2label=id2label,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)

ds_equi = datasets["equestrian"]
train_val_test_equi = ds_equi.train_test_split(test_size=0.2, seed=SEED)
test_ds_equi = train_val_test_equi['test']

ds_wind = datasets["wind_energy"]
train_val_test_wind = ds_wind.train_test_split(test_size=0.2, seed=SEED)
test_ds_wind = train_val_test_wind['test']

error_analysis(
    model_ckpt=os.path.join(MODELS_ROOT, "indomain", "rubert", "equestrian"),
    tokenizer=tokenizer_rubert,
    test_dataset=test_ds_equi,
    domain_name="ruBERT-base In-domain equestrian",
    label2id=label2id, id2label=id2label,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE
)

error_analysis(
    model_ckpt=os.path.join(MODELS_ROOT, "indomain", "ruroberta", "wind_energy"),
    tokenizer=tokenizer_ruroberta,
    test_dataset=test_ds_wind,
    domain_name="ruRoBERTa-large In-domain wind_energy",
    label2id=label2id, id2label=id2label,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/2433 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Анализ ошибок: ruBERT-base LODO heart_failure ---
TP: 803, FP: 1088, FN: 5075

Распределение длин TP:
  длина 1: 543
  длина 2: 208
  длина 3: 49
  длина 4: 3

Распределение длин FP:
  длина 1: 831
  длина 2: 219
  длина 3: 33
  длина 4: 3
  длина 5: 2

Распределение длин FN:
  длина 1: 3303
  длина 2: 1156
  длина 3: 438
  длина 4: 143
  длина 5: 35

Примеры False Positives:
  'выброса левого желудочка'
  'сердечная недостаточность'
  'длинной оси'

Примеры False Negatives:
  'пациентов'
  'фракцию выброса левого желудочка'
  'систолическая функция'


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...


--- Анализ ошибок: ruBERT-base In-domain equestrian ---
TP: 924, FP: 352, FN: 217

Распределение длин TP:
  длина 1: 781
  длина 2: 126
  длина 3: 16
  длина 4: 1

Распределение длин FP:
  длина 1: 285
  длина 2: 52
  длина 3: 10
  длина 4: 5

Распределение длин FN:
  длина 1: 146
  длина 2: 56
  длина 3: 13
  длина 4: 2

Примеры False Positives:
  'седла'
  'направления'
  'грудной'

Примеры False Negatives:
  'сгибания'
  'следовать'
  'грудной клеткой'


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Map:   0%|          | 0/1328 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Анализ ошибок: ruRoBERTa-large In-domain wind_energy ---
TP: 293, FP: 306, FN: 391

Распределение длин TP:
  длина 1: 191
  длина 2: 83
  длина 3: 14
  длина 6: 5

Распределение длин FP:
  длина 1: 233
  длина 2: 65
  длина 3: 5
  длина 4: 3

Распределение длин FN:
  длина 1: 139
  длина 2: 170
  длина 3: 64
  длина 4: 9
  длина 5: 8
  длина 6: 1

Примеры False Positives:
  'УМЕРЕННЫЙ БРИЗ'
  'рынка'
  'Ekostrategija'

Примеры False Negatives:
  'Джонкман'
  'Буль'
  'ветрового потока'
